# Toronto Café Location-Scoring: Neighbourhood Ingest

**Pipeline stage:** Bronze (raw landing)  
**Source:** City of Toronto Open Data, Neighbourhoods (158 areas)  
**Source URL:** https://open.toronto.ca/dataset/neighbourhoods/  
**Destination:** MongoDB `toronto_cafe.neighbourhoods`

This notebook takes the City of Toronto neighbourhood boundary file and lands it in the database as the raw bronze layer. Each of the 158 neighbourhoods becomes one document that keeps its original fields intact, carries its boundary outline for later spatial work, and records where the data came from and when it was pulled.

The neighbourhood code is used as each document's unique key, so the load can be re-run any number of times without creating duplicates.

## 1. Setup and connection

Import the libraries, load the connection string from the local `.env` file, and connect to the project database.

In [ ]:
import os, json                                    # os for env vars, json for parsing the boundary file
from datetime import datetime, timezone            # used to timestamp each record at ingest
from dotenv import load_dotenv                      # reads the local .env so the connection string stays out of the code
from pymongo import MongoClient                     # driver for talking to MongoDB
from datetime import datetime, timezone             # (duplicate of the import above; safe to remove)
load_dotenv(".env")                                 # load MONGODB_URI from .env into the environment
client = MongoClient(os.environ["MONGODB_URI"])     # open the connection to the Atlas cluster
db = client["toronto_cafe"]                         # select the project database (created on first write)

## 2. Load the raw boundary file

Read the neighbourhood boundary file from disk into memory. It comes in as a single structure holding a list of 158 features, one per neighbourhood.

In [1]:
import json

with open("data/raw/Neighbourhoods - 4326.geojson", encoding="utf-8") as f:
    geo = json.load(f)

print(type(geo))                        # what did we get?
print(geo.keys())                       # top-level structure
print("number of features:", len(geo["features"]))

<class 'dict'>
dict_keys(['type', 'name', 'crs', 'features'])
number of features: 158


## 3. Inspect one neighbourhood

Look at a single feature before landing anything, so the shape is clear. Each feature has its descriptive fields under `properties` and its boundary outline under `geometry`.

In [2]:
first = geo["features"][0]
print(first.keys())                     # each feature's parts
print(first["properties"])              # the fields you care about
print("geometry type:", first["geometry"]["type"])

dict_keys(['type', 'properties', 'geometry'])
{'_id': 1, 'AREA_ID': 2502366, 'AREA_ATTR_ID': 26022881, 'PARENT_AREA_ID': None, 'AREA_SHORT_CODE': '174', 'AREA_LONG_CODE': '174', 'AREA_NAME': 'South Eglinton-Davisville', 'AREA_DESC': 'South Eglinton-Davisville (174)', 'CLASSIFICATION': 'Not an NIA or Emerging Neighbourhood', 'CLASSIFICATION_CODE': 'NA', 'OBJECTID': 17824737.0}
geometry type: MultiPolygon


In [ ]:
print(json.dumps(first["properties"], indent=2))   # show one neighbourhood's fields in a readable, indented layout

## 4. Reshape and land into the bronze layer

Walk through all 158 neighbourhoods. For each one, build a document keyed on the neighbourhood code, keep the original fields, attach the boundary shape, and stamp it with its source and pull time. Writing with an upsert on the key means re-running this never creates duplicates.

In [44]:
neighbourhoods = db["neighbourhoods"]              # open (or lazily create) the destination collection

for feature in geo["features"]:                    # walk through all 158 features, one at a time
    props = feature["properties"]                  # grab this feature's descriptive fields

    doc = {                                         # build the document we want to store
        "_id": props["AREA_SHORT_CODE"],           # use the neighbourhood code as the unique key
        "properties": props,                        # keep all the raw fields, untouched (faithful to source)
        "geometry": feature["geometry"],            # the boundary shape — needed for mapping/spatial work later
        "source": "toronto_open_data",              # where this data came from (provenance)
        "ingested_at": datetime.now(timezone.utc),  # when we pulled it (freshness stamp)
    }                                               # <- dict closes here, before the upsert

    neighbourhoods.update_one(                      # save it: update if it exists, insert if it doesn't
        {"_id": doc["_id"]},                        # find any existing doc with this same key
        {"$set": doc},                              # overwrite its contents with our fresh version
        upsert=True,                                # if none exists yet, create it
    )

print("done — processed", len(geo["features"]), "features")   # feedback 

done — processed 158 features


## 5. Verify the load

Confirm that 158 documents landed and check the shape of one.

In [45]:
print(neighbourhoods.count_documents({}))   # count the docs — should be 158
print(neighbourhoods.find_one())            # peek at one landed doc to confirm the shape

158
{'_id': '174', 'geometry': {'type': 'MultiPolygon', 'coordinates': [[[[-79.3863510515018, 43.6978312650188], [-79.3862291956072, 43.6975037220317], [-79.38638812454, 43.6975077326547], [-79.3864562999666, 43.6975192590321], [-79.3866606404473, 43.6975010721977], [-79.3870109672437, 43.6974685463247], [-79.3876455796026, 43.6973977736555], [-79.3895282148369, 43.6969911842496], [-79.3927549645318, 43.6962824893798], [-79.3950210963475, 43.6958404334629], [-79.3951285018801, 43.695814734359], [-79.3952833508039, 43.6957799963262], [-79.3954612917379, 43.6957424798034], [-79.3955690097916, 43.6957223523688], [-79.3957162937643, 43.6957001276303], [-79.3958852874049, 43.6956853665438], [-79.3961081541657, 43.6956666126055], [-79.3962045104245, 43.696127026043], [-79.3964710412413, 43.6973739370436], [-79.3966427138492, 43.6983012068135], [-79.3968612104594, 43.6993320241908], [-79.3969539406084, 43.6997866636074], [-79.3970391853112, 43.7002952858583], [-79.3971592350693, 43.7008534463